# Vesuvius Model Prediction Visualization

## Purpose:
- Visualize model predictions vs ground truth
- Identify issues: over-prediction, under-prediction, connectivity breaks
- Compare different models (V4 vs V5)

## Color Coding:
- **Yellow**: True Positive (both GT and Pred)
- **Cyan/Green**: False Negative (GT only - model missed)
- **Orange/Red**: False Positive (Pred only - model over-predicted)


In [ ]:
!pip install tifffile imagecodecs matplotlib -q

In [ ]:
import os
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from typing import Tuple, List, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================================
# CONFIGURATION - V4 MODEL (Val Dice = 0.63)
# =============================================================================
class Config:
    # Data paths - UPDATE THESE
    DATA_ROOT = Path("/kaggle/input/3d-volume-training-data")
    TRAIN_IMAGES = DATA_ROOT / "train_images"
    TRAIN_LABELS = DATA_ROOT / "train_labels"
    
    # Model checkpoint - V4 MODEL (better val dice = 0.63)
    CHECKPOINT_PATH = Path("/kaggle/input/v4-model/checkpoints_v4/best_model.pth")
    
    # Model architecture - V4 uses 128³ patches
    PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)  # V4: 128³
    FEATURES: List[int] = [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = [1, 3, 4, 6, 6, 6]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # Inference settings
    OVERLAP: float = 0.5
    THRESHOLD: float = 0.4
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True

cfg = Config()
print(f"Model: V4 (Val Dice = 0.63)")
print(f"Patch size: {cfg.PATCH_SIZE}")
print(f"Checkpoint: {cfg.CHECKPOINT_PATH}")

In [ ]:
# =============================================================================
# MODEL ARCHITECTURE (SafeInstanceNorm3d)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super(SafeInstanceNorm3d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        if self.affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)

    def forward(self, x):
        mean = x.mean(dim=[2, 3, 4], keepdim=True)
        var = x.var(dim=[2, 3, 4], keepdim=True, unbiased=False)
        var_safe = torch.clamp(var, min=self.eps)
        x_norm = (x - mean) / torch.sqrt(var_safe + self.eps)
        if self.affine:
            x_norm = self.weight.view(1, -1, 1, 1, 1) * x_norm + self.bias.view(1, -1, 1, 1, 1)
        return x_norm


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),
            SafeInstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels, channels) for _ in range(n_convs)])
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)

    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None, use_scse=True, use_deep_supervision=True):
        super().__init__()
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)
        
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(ConvBlock(in_channels, feat), *[ResBlock(feat, n_convs=2) for _ in range(nb)])
            self.encoders.append(encoder)
            self.attentions.append(scSEBlock(feat) if use_scse else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2, bias=True))
        
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2, bias=True))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))
        
        self.final = nn.Conv3d(features[0], out_ch, 1, bias=True)
        
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(4, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1, bias=True))

    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        
        return self.final(x)


print("Model architecture defined")

In [ ]:
# =============================================================================
# INFERENCE FUNCTIONS
# =============================================================================

def normalize_volume(volume: np.ndarray) -> np.ndarray:
    vol = volume.astype(np.float32)
    return (vol - vol.mean()) / (vol.std() + 1e-8)


@torch.no_grad()
def predict_volume(model, volume, patch_size, overlap=0.5, device="cuda", use_amp=True):
    """Run sliding window inference on a volume."""
    model.eval()
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd * (1 - overlap)), int(ph * (1 - overlap)), int(pw * (1 - overlap))
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Gaussian weighting
    sigma = 0.125
    gauss_z = np.exp(-0.5 * ((np.arange(pd) - pd/2) / (pd * sigma)) ** 2)
    gauss_y = np.exp(-0.5 * ((np.arange(ph) - ph/2) / (ph * sigma)) ** 2)
    gauss_x = np.exp(-0.5 * ((np.arange(pw) - pw/2) / (pw * sigma)) ** 2)
    gauss_weight = gauss_z[:, None, None] * gauss_y[None, :, None] * gauss_x[None, None, :]
    gauss_weight = gauss_weight.astype(np.float32)
    
    # Generate positions
    z_pos = list(range(0, max(1, D - pd + 1), sd))
    if D > pd and (D - pd) not in z_pos:
        z_pos.append(D - pd)
    y_pos = list(range(0, max(1, H - ph + 1), sh))
    if H > ph and (H - ph) not in y_pos:
        y_pos.append(H - ph)
    x_pos = list(range(0, max(1, W - pw + 1), sw))
    if W > pw and (W - pw) not in x_pos:
        x_pos.append(W - pw)
    
    vol_norm = normalize_volume(volume)
    
    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = vol_norm[z:z+pd, y:y+ph, x:x+pw].copy()
                actual_shape = patch.shape
                
                if patch.shape != (pd, ph, pw):
                    pad = [(0, max(0, pd - patch.shape[0])),
                           (0, max(0, ph - patch.shape[1])),
                           (0, max(0, pw - patch.shape[2]))]
                    patch = np.pad(patch, pad, mode='reflect')
                
                patch = patch.astype(np.float32)
                inp = torch.from_numpy(patch[None, None]).float().to(device)
                
                with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                    out = model(inp)
                    if isinstance(out, dict):
                        out = out['output']
                    out = torch.sigmoid(out).squeeze().cpu().numpy()
                
                out = out[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                weight = gauss_weight[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                
                pred_sum[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += out * weight
                count[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += weight
    
    return pred_sum / np.maximum(count, 1e-8)


print("Inference functions defined")

In [ ]:
# =============================================================================
# VISUALIZATION FUNCTIONS
# =============================================================================

def create_overlay_image(ct_slice, gt_slice, pred_slice):
    """
    Create an overlay visualization.
    
    Returns RGB image where:
    - Yellow: True Positive (GT=1 and Pred=1)
    - Cyan: False Negative (GT=1 and Pred=0) - model missed
    - Red/Orange: False Positive (GT=0 and Pred=1) - model over-predicted
    - Gray: Background CT
    """
    # Normalize CT for display
    ct_norm = (ct_slice - ct_slice.min()) / (ct_slice.max() - ct_slice.min() + 1e-8)
    
    # Create RGB image from CT (grayscale)
    rgb = np.stack([ct_norm, ct_norm, ct_norm], axis=-1)
    
    # Compute TP, FN, FP
    gt_bool = gt_slice > 0
    pred_bool = pred_slice > 0
    
    tp = gt_bool & pred_bool      # True Positive
    fn = gt_bool & ~pred_bool     # False Negative (missed)
    fp = ~gt_bool & pred_bool     # False Positive (over-prediction)
    
    # Apply colors
    # Yellow for TP
    rgb[tp] = [1.0, 1.0, 0.0]
    # Cyan for FN (what model missed)
    rgb[fn] = [0.0, 1.0, 1.0]
    # Orange/Red for FP (over-prediction)
    rgb[fp] = [1.0, 0.5, 0.0]
    
    return rgb


def visualize_slice(ct_vol, gt_vol, pred_prob, pred_binary, slice_idx, axis='z', 
                    threshold=0.4, figsize=(20, 5)):
    """
    Visualize a single slice with multiple views.
    
    Args:
        ct_vol: CT volume
        gt_vol: Ground truth labels
        pred_prob: Prediction probability map
        pred_binary: Thresholded predictions
        slice_idx: Which slice to show
        axis: 'z' (axial), 'y' (coronal), 'x' (sagittal)
        threshold: Threshold used for pred_binary
    """
    # Get slices based on axis
    if axis == 'z':
        ct_slice = ct_vol[slice_idx, :, :]
        gt_slice = gt_vol[slice_idx, :, :]
        prob_slice = pred_prob[slice_idx, :, :]
        pred_slice = pred_binary[slice_idx, :, :]
        axis_name = 'axial'
    elif axis == 'y':
        ct_slice = ct_vol[:, slice_idx, :]
        gt_slice = gt_vol[:, slice_idx, :]
        prob_slice = pred_prob[:, slice_idx, :]
        pred_slice = pred_binary[:, slice_idx, :]
        axis_name = 'coronal'
    else:  # axis == 'x'
        ct_slice = ct_vol[:, :, slice_idx]
        gt_slice = gt_vol[:, :, slice_idx]
        prob_slice = pred_prob[:, :, slice_idx]
        pred_slice = pred_binary[:, :, slice_idx]
        axis_name = 'sagittal'
    
    # Create overlay
    overlay = create_overlay_image(ct_slice, gt_slice, pred_slice)
    
    # Create figure
    fig, axes = plt.subplots(1, 5, figsize=figsize)
    
    # 1. CT image
    axes[0].imshow(ct_slice, cmap='gray')
    axes[0].set_title(f'CT ({axis_name} {axis}={slice_idx})')
    axes[0].axis('off')
    
    # 2. Ground Truth
    axes[1].imshow(ct_slice, cmap='gray')
    axes[1].imshow(gt_slice, cmap='Greens', alpha=0.5 * (gt_slice > 0))
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')
    
    # 3. Prediction Probability
    axes[2].imshow(prob_slice, cmap='hot', vmin=0, vmax=1)
    axes[2].set_title(f'Pred Prob (thr={threshold})')
    axes[2].axis('off')
    
    # 4. Binary Prediction
    axes[3].imshow(ct_slice, cmap='gray')
    axes[3].imshow(pred_slice, cmap='Oranges', alpha=0.5 * (pred_slice > 0))
    axes[3].set_title('Our Prediction')
    axes[3].axis('off')
    
    # 5. Overlay (TP/FN/FP)
    axes[4].imshow(overlay)
    axes[4].set_title('Overlay (Y=TP, C=FN, O=FP)')
    axes[4].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    gt_bool = gt_slice > 0
    pred_bool = pred_slice > 0
    tp = (gt_bool & pred_bool).sum()
    fn = (gt_bool & ~pred_bool).sum()
    fp = (~gt_bool & pred_bool).sum()
    
    dice = (2 * tp) / (2 * tp + fn + fp + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    
    print(f"Slice {slice_idx} Stats:")
    print(f"  TP (Yellow): {tp:,}")
    print(f"  FN (Cyan - missed): {fn:,}")
    print(f"  FP (Orange - over-pred): {fp:,}")
    print(f"  Dice: {dice:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")


def visualize_volume_summary(ct_vol, gt_vol, pred_binary, n_slices=5, axis='z'):
    """
    Show multiple slices from the volume.
    """
    if axis == 'z':
        max_idx = ct_vol.shape[0]
    elif axis == 'y':
        max_idx = ct_vol.shape[1]
    else:
        max_idx = ct_vol.shape[2]
    
    # Get evenly spaced slices
    slice_indices = np.linspace(max_idx * 0.2, max_idx * 0.8, n_slices, dtype=int)
    
    fig, axes = plt.subplots(n_slices, 3, figsize=(15, 4 * n_slices))
    
    for i, slice_idx in enumerate(slice_indices):
        if axis == 'z':
            ct_slice = ct_vol[slice_idx, :, :]
            gt_slice = gt_vol[slice_idx, :, :]
            pred_slice = pred_binary[slice_idx, :, :]
        elif axis == 'y':
            ct_slice = ct_vol[:, slice_idx, :]
            gt_slice = gt_vol[:, slice_idx, :]
            pred_slice = pred_binary[:, slice_idx, :]
        else:
            ct_slice = ct_vol[:, :, slice_idx]
            gt_slice = gt_vol[:, :, slice_idx]
            pred_slice = pred_binary[:, :, slice_idx]
        
        overlay = create_overlay_image(ct_slice, gt_slice, pred_slice)
        
        # GT
        axes[i, 0].imshow(ct_slice, cmap='gray')
        axes[i, 0].imshow(gt_slice, cmap='Greens', alpha=0.5 * (gt_slice > 0))
        axes[i, 0].set_title(f'GT ({axis}={slice_idx})')
        axes[i, 0].axis('off')
        
        # Prediction
        axes[i, 1].imshow(ct_slice, cmap='gray')
        axes[i, 1].imshow(pred_slice, cmap='Oranges', alpha=0.5 * (pred_slice > 0))
        axes[i, 1].set_title(f'Pred ({axis}={slice_idx})')
        axes[i, 1].axis('off')
        
        # Overlay
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(f'Overlay ({axis}={slice_idx})')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()


print("Visualization functions defined")

In [ ]:
# =============================================================================
# LOAD MODEL
# =============================================================================

print("Loading model...")
model = ResEncUNet3D(
    features=cfg.FEATURES,
    n_blocks=cfg.N_BLOCKS,
    use_scse=cfg.USE_SCSE,
    use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
)

# Load checkpoint
checkpoint = torch.load(cfg.CHECKPOINT_PATH, map_location=cfg.DEVICE, weights_only=False)

if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
    if 'best_score' in checkpoint:
        print(f"  Best Val Dice: {checkpoint['best_score']:.4f}")
    if 'epoch' in checkpoint:
        print(f"  Epoch: {checkpoint['epoch']}")
else:
    state_dict = checkpoint

# Clean keys
cleaned_state = {}
for k, v in state_dict.items():
    key = k.replace('module.', '').replace('_orig_mod.', '')
    cleaned_state[key] = v

model.load_state_dict(cleaned_state, strict=False)
model = model.to(cfg.DEVICE).eval()

print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

In [ ]:
# =============================================================================
# SELECT A SAMPLE TO VISUALIZE
# =============================================================================

import pandas as pd

# List available volumes
train_csv = cfg.DATA_ROOT / "train.csv"
df = pd.read_csv(train_csv)

# Get list of available volumes
available = []
for vol_id in df['id'].values[:20]:  # Check first 20
    img_path = cfg.TRAIN_IMAGES / f"{vol_id}.tif"
    lbl_path = cfg.TRAIN_LABELS / f"{vol_id}.tif"
    if img_path.exists() and lbl_path.exists():
        available.append(vol_id)

print(f"Available volumes (first 20): {len(available)}")
print(available[:10])

# SELECT VOLUME TO VISUALIZE - CHANGE THIS
VOLUME_ID = available[0] if available else None
print(f"\nSelected volume: {VOLUME_ID}")

In [ ]:
# =============================================================================
# LOAD DATA AND RUN INFERENCE
# =============================================================================

if VOLUME_ID is not None:
    print(f"Loading volume {VOLUME_ID}...")
    
    # Load CT and GT
    ct_vol = tifffile.imread(str(cfg.TRAIN_IMAGES / f"{VOLUME_ID}.tif"))
    gt_vol = tifffile.imread(str(cfg.TRAIN_LABELS / f"{VOLUME_ID}.tif"))
    
    print(f"CT shape: {ct_vol.shape}")
    print(f"GT shape: {gt_vol.shape}")
    print(f"GT unique values: {np.unique(gt_vol)}")
    
    # Run inference
    print(f"\nRunning inference with patch size {cfg.PATCH_SIZE}...")
    pred_prob = predict_volume(
        model, ct_vol, 
        patch_size=cfg.PATCH_SIZE,
        overlap=cfg.OVERLAP,
        device=cfg.DEVICE,
        use_amp=cfg.USE_AMP
    )
    
    # Threshold
    pred_binary = (pred_prob > cfg.THRESHOLD).astype(np.uint8)
    
    print(f"\nPrediction stats:")
    print(f"  Prob range: [{pred_prob.min():.4f}, {pred_prob.max():.4f}]")
    print(f"  FG% (threshold={cfg.THRESHOLD}): {100 * pred_binary.mean():.3f}%")
    print(f"  GT FG%: {100 * (gt_vol == 1).mean():.3f}%")
else:
    print("No volume selected!")

In [ ]:
# =============================================================================
# CALCULATE OVERALL METRICS
# =============================================================================

if VOLUME_ID is not None:
    # Consider only valid labels (ignore label=2)
    valid_mask = gt_vol != 2
    gt_fg = (gt_vol == 1) & valid_mask
    pred_fg = (pred_binary == 1) & valid_mask
    
    tp = (gt_fg & pred_fg).sum()
    fn = (gt_fg & ~pred_fg).sum()
    fp = (~gt_fg & pred_fg).sum()
    
    dice = (2 * tp) / (2 * tp + fn + fp + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    
    print("="*60)
    print(f"VOLUME {VOLUME_ID} - OVERALL METRICS")
    print("="*60)
    print(f"True Positives:  {tp:,}")
    print(f"False Negatives: {fn:,} (model missed)")
    print(f"False Positives: {fp:,} (over-prediction)")
    print(f"")
    print(f"Dice Score:  {dice:.4f}")
    print(f"Precision:   {precision:.4f}")
    print(f"Recall:      {recall:.4f}")
    print("="*60)

In [ ]:
# =============================================================================
# VISUALIZE MULTIPLE SLICES
# =============================================================================

if VOLUME_ID is not None:
    print("Visualizing axial slices (z-axis)...")
    visualize_volume_summary(ct_vol, gt_vol, pred_binary, n_slices=5, axis='z')

In [ ]:
# =============================================================================
# VISUALIZE SPECIFIC SLICE WITH ALL VIEWS
# =============================================================================

if VOLUME_ID is not None:
    # Choose a slice index (middle of volume)
    slice_z = ct_vol.shape[0] // 2
    
    print(f"\nDetailed view of slice z={slice_z}:")
    visualize_slice(ct_vol, gt_vol, pred_prob, pred_binary, 
                    slice_idx=slice_z, axis='z', threshold=cfg.THRESHOLD)

In [ ]:
# =============================================================================
# INTERACTIVE: TRY DIFFERENT THRESHOLDS
# =============================================================================

if VOLUME_ID is not None:
    print("Comparing different thresholds...")
    
    slice_idx = ct_vol.shape[0] // 2
    thresholds = [0.3, 0.4, 0.5, 0.6]
    
    fig, axes = plt.subplots(len(thresholds), 3, figsize=(15, 4 * len(thresholds)))
    
    for i, thr in enumerate(thresholds):
        pred_thr = (pred_prob > thr).astype(np.uint8)
        
        ct_slice = ct_vol[slice_idx, :, :]
        gt_slice = gt_vol[slice_idx, :, :]
        pred_slice = pred_thr[slice_idx, :, :]
        
        overlay = create_overlay_image(ct_slice, gt_slice, pred_slice)
        
        # Pred
        axes[i, 0].imshow(ct_slice, cmap='gray')
        axes[i, 0].imshow(pred_slice, cmap='Oranges', alpha=0.5 * (pred_slice > 0))
        axes[i, 0].set_title(f'Threshold={thr}')
        axes[i, 0].axis('off')
        
        # Overlay
        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title(f'Overlay (thr={thr})')
        axes[i, 1].axis('off')
        
        # Stats
        gt_bool = gt_slice > 0
        pred_bool = pred_slice > 0
        tp = (gt_bool & pred_bool).sum()
        fn = (gt_bool & ~pred_bool).sum()
        fp = (~gt_bool & pred_bool).sum()
        dice = (2 * tp) / (2 * tp + fn + fp + 1e-8)
        
        axes[i, 2].text(0.5, 0.5, f"Threshold: {thr}\n\nDice: {dice:.4f}\nTP: {tp:,}\nFN: {fn:,}\nFP: {fp:,}",
                       ha='center', va='center', fontsize=14, transform=axes[i, 2].transAxes)
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# ANALYZE: WHERE IS THE MODEL FAILING?
# =============================================================================

if VOLUME_ID is not None:
    print("Analyzing model failures...")
    
    # Find slices with most errors
    valid_mask = gt_vol != 2
    gt_fg = (gt_vol == 1) & valid_mask
    pred_fg = (pred_binary == 1) & valid_mask
    
    fn_per_slice = []
    fp_per_slice = []
    
    for z in range(gt_vol.shape[0]):
        fn = (gt_fg[z] & ~pred_fg[z]).sum()
        fp = (~gt_fg[z] & pred_fg[z]).sum()
        fn_per_slice.append(fn)
        fp_per_slice.append(fp)
    
    fn_per_slice = np.array(fn_per_slice)
    fp_per_slice = np.array(fp_per_slice)
    
    # Plot error distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].plot(fn_per_slice, label='False Negatives (missed)', color='cyan')
    axes[0].plot(fp_per_slice, label='False Positives (over-pred)', color='orange')
    axes[0].set_xlabel('Slice (z)')
    axes[0].set_ylabel('Error count')
    axes[0].set_title('Errors per Slice')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Show worst slices
    worst_fn_slices = np.argsort(fn_per_slice)[-5:][::-1]
    worst_fp_slices = np.argsort(fp_per_slice)[-5:][::-1]
    
    axes[1].barh(range(5), fn_per_slice[worst_fn_slices], alpha=0.7, label='FN', color='cyan')
    axes[1].set_yticks(range(5))
    axes[1].set_yticklabels([f'z={s}' for s in worst_fn_slices])
    axes[1].set_xlabel('Error count')
    axes[1].set_title('Slices with Most False Negatives')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nWorst FN slices: {worst_fn_slices}")
    print(f"Worst FP slices: {worst_fp_slices}")

In [ ]:
# =============================================================================
# VISUALIZE WORST SLICE
# =============================================================================

if VOLUME_ID is not None:
    # Show the slice with most false negatives
    worst_slice = worst_fn_slices[0]
    print(f"\nVisualizing worst slice (z={worst_slice}) with most false negatives:")
    visualize_slice(ct_vol, gt_vol, pred_prob, pred_binary, 
                    slice_idx=worst_slice, axis='z', threshold=cfg.THRESHOLD)

In [ ]:
# =============================================================================
# THRESHOLD SWEEP - FIND OPTIMAL THRESHOLD
# =============================================================================

if VOLUME_ID is not None:
    print("Analyzing threshold impact on metrics...")
    
    # Consider only valid labels
    valid_mask = gt_vol != 2
    gt_fg = (gt_vol == 1) & valid_mask
    
    thresholds = np.arange(0.1, 0.8, 0.05)
    metrics = {'threshold': [], 'dice': [], 'precision': [], 'recall': [], 'fg_pct': []}
    
    for thr in thresholds:
        pred_fg = (pred_prob > thr) & valid_mask
        
        tp = (gt_fg & pred_fg).sum()
        fn = (gt_fg & ~pred_fg).sum()
        fp = (~gt_fg & pred_fg).sum()
        
        dice = (2 * tp) / (2 * tp + fn + fp + 1e-8)
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        fg_pct = pred_fg.sum() / valid_mask.sum() * 100
        
        metrics['threshold'].append(thr)
        metrics['dice'].append(dice)
        metrics['precision'].append(precision)
        metrics['recall'].append(recall)
        metrics['fg_pct'].append(fg_pct)
    
    # Find optimal threshold
    best_idx = np.argmax(metrics['dice'])
    best_thr = metrics['threshold'][best_idx]
    best_dice = metrics['dice'][best_idx]
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Metrics vs threshold
    axes[0].plot(metrics['threshold'], metrics['dice'], 'g-o', label='Dice', linewidth=2)
    axes[0].plot(metrics['threshold'], metrics['precision'], 'b-s', label='Precision', linewidth=2)
    axes[0].plot(metrics['threshold'], metrics['recall'], 'r-^', label='Recall', linewidth=2)
    axes[0].axvline(x=best_thr, color='green', linestyle='--', alpha=0.7, label=f'Best thr={best_thr:.2f}')
    axes[0].axvline(x=0.4, color='gray', linestyle=':', alpha=0.7, label='Current thr=0.4')
    axes[0].set_xlabel('Threshold', fontsize=12)
    axes[0].set_ylabel('Score', fontsize=12)
    axes[0].set_title('Metrics vs Threshold', fontsize=14)
    axes[0].legend(loc='best')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1])
    
    # FG% vs threshold
    axes[1].plot(metrics['threshold'], metrics['fg_pct'], 'purple', linewidth=2, marker='o')
    gt_fg_pct = gt_fg.sum() / valid_mask.sum() * 100
    axes[1].axhline(y=gt_fg_pct, color='green', linestyle='--', label=f'GT FG%={gt_fg_pct:.2f}%')
    axes[1].axvline(x=best_thr, color='green', linestyle='--', alpha=0.7)
    axes[1].set_xlabel('Threshold', fontsize=12)
    axes[1].set_ylabel('Foreground %', fontsize=12)
    axes[1].set_title('Prediction FG% vs Threshold', fontsize=14)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary table
    print("\n" + "="*70)
    print("THRESHOLD ANALYSIS SUMMARY")
    print("="*70)
    print(f"{'Threshold':>10} {'Dice':>10} {'Precision':>10} {'Recall':>10} {'FG%':>10}")
    print("-"*70)
    for i, thr in enumerate(metrics['threshold']):
        marker = " <-- BEST" if i == best_idx else ""
        marker = " <-- CURRENT" if abs(thr - 0.4) < 0.01 else marker
        print(f"{thr:>10.2f} {metrics['dice'][i]:>10.4f} {metrics['precision'][i]:>10.4f} {metrics['recall'][i]:>10.4f} {metrics['fg_pct'][i]:>10.2f}%{marker}")
    print("="*70)
    print(f"\nOptimal threshold: {best_thr:.2f} with Dice={best_dice:.4f}")
    print(f"GT FG%: {gt_fg_pct:.2f}%")
    
    # Recommendation
    if best_thr < 0.4:
        print(f"\n⚠️  RECOMMENDATION: Lower threshold from 0.4 to {best_thr:.2f} for better Dice score")
        print(f"    This would improve recall from {metrics['recall'][8]:.4f} to {metrics['recall'][best_idx]:.4f}")

In [ ]:
# =============================================================================
# VISUALIZE WORST SLICE
# =============================================================================

if VOLUME_ID is not None:
    # Show the slice with most false negatives
    worst_slice = worst_fn_slices[0]
    print(f"\nVisualizing worst slice (z={worst_slice}) with most false negatives:")
    visualize_slice(ct_vol, gt_vol, pred_prob, pred_binary, 
                    slice_idx=worst_slice, axis='z', threshold=cfg.THRESHOLD)

In [ ]:
# =============================================================================
# PROBABILITY DISTRIBUTION ANALYSIS
# =============================================================================

if VOLUME_ID is not None:
    print("Analyzing prediction probability distribution...")
    
    # Sample probabilities (too many to plot all)
    valid_mask = gt_vol != 2
    gt_fg_mask = (gt_vol == 1) & valid_mask
    gt_bg_mask = (gt_vol == 0) & valid_mask
    
    # Get probability values
    fg_probs = pred_prob[gt_fg_mask]
    bg_probs = pred_prob[gt_bg_mask]
    
    # Sample for plotting (otherwise too many points)
    n_samples = min(100000, len(fg_probs), len(bg_probs))
    fg_sample = np.random.choice(fg_probs, size=n_samples, replace=False) if len(fg_probs) > n_samples else fg_probs
    bg_sample = np.random.choice(bg_probs, size=n_samples, replace=False) if len(bg_probs) > n_samples else bg_probs
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(bg_sample, bins=50, alpha=0.5, label='Background (GT=0)', color='blue', density=True)
    axes[0].hist(fg_sample, bins=50, alpha=0.5, label='Foreground (GT=1)', color='red', density=True)
    axes[0].axvline(x=0.4, color='gray', linestyle='--', linewidth=2, label='Current thr=0.4')
    axes[0].axvline(x=0.2, color='green', linestyle=':', linewidth=2, label='Lower thr=0.2')
    axes[0].set_xlabel('Prediction Probability', fontsize=12)
    axes[0].set_ylabel('Density', fontsize=12)
    axes[0].set_title('Probability Distribution by GT Class', fontsize=14)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Box plot
    data = [bg_sample, fg_sample]
    bp = axes[1].boxplot(data, labels=['Background', 'Foreground'], patch_artist=True)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][1].set_facecolor('lightcoral')
    axes[1].axhline(y=0.4, color='gray', linestyle='--', linewidth=2, label='Current thr=0.4')
    axes[1].axhline(y=0.2, color='green', linestyle=':', linewidth=2, label='Lower thr=0.2')
    axes[1].set_ylabel('Prediction Probability', fontsize=12)
    axes[1].set_title('Probability Distribution Box Plot', fontsize=14)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print stats
    print(f"\nProbability Statistics:")
    print(f"  Foreground (GT=1) - mean: {fg_probs.mean():.4f}, median: {np.median(fg_probs):.4f}, std: {fg_probs.std():.4f}")
    print(f"  Background (GT=0) - mean: {bg_probs.mean():.4f}, median: {np.median(bg_probs):.4f}, std: {bg_probs.std():.4f}")
    print(f"\n  % of FG with prob > 0.4: {100 * (fg_probs > 0.4).mean():.1f}%")
    print(f"  % of FG with prob > 0.3: {100 * (fg_probs > 0.3).mean():.1f}%")
    print(f"  % of FG with prob > 0.2: {100 * (fg_probs > 0.2).mean():.1f}%")
    print(f"  % of FG with prob > 0.1: {100 * (fg_probs > 0.1).mean():.1f}%")
    
    if fg_probs.mean() < 0.4:
        print(f"\n⚠️  Model is producing low confidence predictions for foreground voxels!")
        print(f"    Mean FG probability ({fg_probs.mean():.3f}) is below threshold 0.4")
        print(f"    Consider lowering threshold or continuing training")

In [ ]:
# =============================================================================
# MORPHOLOGICAL DILATION - FILL THIN PREDICTIONS
# =============================================================================
# The model predicts thin edge lines instead of filled surfaces.
# We'll use morphological dilation to thicken the predictions.

from scipy.ndimage import binary_dilation, generate_binary_structure

def apply_dilation_3d(binary_mask, iterations=1, structure_type='ball'):
    """
    Apply 3D morphological dilation to thicken predictions.
    
    Args:
        binary_mask: Binary prediction mask (D, H, W)
        iterations: Number of dilation iterations (more = thicker)
        structure_type: 'ball' (sphere), 'cross' (6-connected), 'cube' (26-connected)
    
    Returns:
        Dilated binary mask
    """
    if structure_type == 'cross':
        # 6-connected (faces only)
        struct = generate_binary_structure(3, 1)
    elif structure_type == 'cube':
        # 26-connected (faces + edges + corners)
        struct = generate_binary_structure(3, 3)
    elif structure_type == 'ball':
        # Approximate sphere using larger structure
        struct = np.ones((3, 3, 3), dtype=bool)
    else:
        struct = generate_binary_structure(3, 1)
    
    dilated = binary_dilation(binary_mask, structure=struct, iterations=iterations)
    return dilated.astype(np.uint8)


def apply_dilation_2d_per_slice(binary_mask, iterations=1):
    """
    Apply 2D dilation to each slice independently.
    This preserves the layered structure of papyrus better.
    
    Args:
        binary_mask: Binary prediction mask (D, H, W)
        iterations: Number of dilation iterations
    
    Returns:
        Dilated binary mask
    """
    from scipy.ndimage import binary_dilation as bd2d
    
    result = np.zeros_like(binary_mask)
    struct_2d = np.ones((3, 3), dtype=bool)  # 8-connected 2D
    
    for z in range(binary_mask.shape[0]):
        result[z] = bd2d(binary_mask[z], structure=struct_2d, iterations=iterations)
    
    return result.astype(np.uint8)


print("Morphological dilation functions defined")
print("  - apply_dilation_3d(): 3D dilation (thickens in all directions)")
print("  - apply_dilation_2d_per_slice(): 2D dilation per slice (preserves layers)")

In [ ]:
# =============================================================================
# EXPERIMENT: COMPARE DIFFERENT DILATION SETTINGS
# =============================================================================

if VOLUME_ID is not None:
    print("Comparing dilation settings...")
    
    # Use a lower threshold first (0.1 was best)
    base_threshold = 0.1
    pred_base = (pred_prob > base_threshold).astype(np.uint8)
    
    # Test different dilation iterations
    dilation_settings = [
        ("No dilation", pred_base),
        ("2D dilation, iter=1", apply_dilation_2d_per_slice(pred_base, iterations=1)),
        ("2D dilation, iter=2", apply_dilation_2d_per_slice(pred_base, iterations=2)),
        ("2D dilation, iter=3", apply_dilation_2d_per_slice(pred_base, iterations=3)),
        ("3D dilation, iter=1", apply_dilation_3d(pred_base, iterations=1)),
        ("3D dilation, iter=2", apply_dilation_3d(pred_base, iterations=2)),
    ]
    
    # Calculate metrics for each
    valid_mask = gt_vol != 2
    gt_fg = (gt_vol == 1) & valid_mask
    
    print(f"\nBase threshold: {base_threshold}")
    print("="*80)
    print(f"{'Setting':<25} {'Dice':>10} {'Precision':>10} {'Recall':>10} {'FG voxels':>15}")
    print("-"*80)
    
    results = []
    for name, pred in dilation_settings:
        pred_fg = (pred == 1) & valid_mask
        
        tp = (gt_fg & pred_fg).sum()
        fn = (gt_fg & ~pred_fg).sum()
        fp = (~gt_fg & pred_fg).sum()
        
        dice = (2 * tp) / (2 * tp + fn + fp + 1e-8)
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        fg_count = pred_fg.sum()
        
        results.append({
            'name': name,
            'pred': pred,
            'dice': dice,
            'precision': precision,
            'recall': recall,
            'fg_count': fg_count
        })
        
        print(f"{name:<25} {dice:>10.4f} {precision:>10.4f} {recall:>10.4f} {fg_count:>15,}")
    
    print("="*80)
    
    # Find best
    best_result = max(results, key=lambda x: x['dice'])
    print(f"\n✅ BEST: {best_result['name']} with Dice={best_result['dice']:.4f}")

In [ ]:
# =============================================================================
# VISUALIZE DILATION EFFECT ON A SINGLE SLICE
# =============================================================================

if VOLUME_ID is not None:
    print("Visualizing dilation effect...")
    
    slice_idx = ct_vol.shape[0] // 2  # Middle slice
    base_threshold = 0.1
    
    # Get base prediction
    pred_base = (pred_prob > base_threshold).astype(np.uint8)
    
    # Apply different dilations
    pred_2d_1 = apply_dilation_2d_per_slice(pred_base, iterations=1)
    pred_2d_2 = apply_dilation_2d_per_slice(pred_base, iterations=2)
    pred_2d_3 = apply_dilation_2d_per_slice(pred_base, iterations=3)
    pred_3d_1 = apply_dilation_3d(pred_base, iterations=1)
    pred_3d_2 = apply_dilation_3d(pred_base, iterations=2)
    
    # Create visualization
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    
    ct_slice = ct_vol[slice_idx]
    gt_slice = gt_vol[slice_idx]
    
    # Row 1: GT, Base prediction, 2D dilation iter=1, 2D dilation iter=2
    preds_row1 = [
        ("Ground Truth", gt_slice),
        (f"Base (thr={base_threshold})", pred_base[slice_idx]),
        ("2D Dilation iter=1", pred_2d_1[slice_idx]),
        ("2D Dilation iter=2", pred_2d_2[slice_idx]),
    ]
    
    for j, (title, pred_slice) in enumerate(preds_row1):
        if j == 0:
            axes[0, j].imshow(ct_slice, cmap='gray')
            axes[0, j].imshow(pred_slice, cmap='Greens', alpha=0.5 * (pred_slice > 0))
        else:
            axes[0, j].imshow(ct_slice, cmap='gray')
            axes[0, j].imshow(pred_slice, cmap='Oranges', alpha=0.5 * (pred_slice > 0))
        axes[0, j].set_title(title)
        axes[0, j].axis('off')
    
    # Row 2: 2D iter=3, 3D iter=1, 3D iter=2, Empty/Stats
    preds_row2 = [
        ("2D Dilation iter=3", pred_2d_3[slice_idx]),
        ("3D Dilation iter=1", pred_3d_1[slice_idx]),
        ("3D Dilation iter=2", pred_3d_2[slice_idx]),
    ]
    
    for j, (title, pred_slice) in enumerate(preds_row2):
        axes[1, j].imshow(ct_slice, cmap='gray')
        axes[1, j].imshow(pred_slice, cmap='Oranges', alpha=0.5 * (pred_slice > 0))
        axes[1, j].set_title(title)
        axes[1, j].axis('off')
    
    axes[1, 3].axis('off')
    
    # Row 3: Overlays for comparison
    overlays = [
        ("Overlay: Base", create_overlay_image(ct_slice, gt_slice, pred_base[slice_idx])),
        ("Overlay: 2D iter=2", create_overlay_image(ct_slice, gt_slice, pred_2d_2[slice_idx])),
        ("Overlay: 2D iter=3", create_overlay_image(ct_slice, gt_slice, pred_2d_3[slice_idx])),
        ("Overlay: 3D iter=2", create_overlay_image(ct_slice, gt_slice, pred_3d_2[slice_idx])),
    ]
    
    for j, (title, overlay) in enumerate(overlays):
        axes[2, j].imshow(overlay)
        axes[2, j].set_title(title)
        axes[2, j].axis('off')
    
    plt.suptitle(f"Dilation Effect on Slice z={slice_idx} (Yellow=TP, Cyan=FN, Orange=FP)", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# GRID SEARCH: THRESHOLD + DILATION COMBINATIONS
# =============================================================================

if VOLUME_ID is not None:
    print("Grid search: Finding optimal threshold + dilation combination...")
    
    valid_mask = gt_vol != 2
    gt_fg = (gt_vol == 1) & valid_mask
    
    # Test combinations
    thresholds = [0.05, 0.1, 0.15, 0.2, 0.25]
    dilation_iters = [0, 1, 2, 3, 4, 5]  # 0 = no dilation
    
    results_grid = []
    
    print(f"\n{'Threshold':<12} {'Dilation':<12} {'Dice':>10} {'Precision':>10} {'Recall':>10}")
    print("-"*60)
    
    best_dice = 0
    best_config = None
    
    for thr in thresholds:
        pred_base = (pred_prob > thr).astype(np.uint8)
        
        for dil_iter in dilation_iters:
            if dil_iter == 0:
                pred_final = pred_base
            else:
                pred_final = apply_dilation_2d_per_slice(pred_base, iterations=dil_iter)
            
            pred_fg = (pred_final == 1) & valid_mask
            
            tp = (gt_fg & pred_fg).sum()
            fn = (gt_fg & ~pred_fg).sum()
            fp = (~gt_fg & pred_fg).sum()
            
            dice = (2 * tp) / (2 * tp + fn + fp + 1e-8)
            precision = tp / (tp + fp + 1e-8)
            recall = tp / (tp + fn + 1e-8)
            
            results_grid.append({
                'threshold': thr,
                'dilation': dil_iter,
                'dice': dice,
                'precision': precision,
                'recall': recall
            })
            
            marker = ""
            if dice > best_dice:
                best_dice = dice
                best_config = (thr, dil_iter)
                marker = " <-- BEST"
            
            print(f"{thr:<12.2f} {dil_iter:<12} {dice:>10.4f} {precision:>10.4f} {recall:>10.4f}{marker}")
    
    print("="*60)
    print(f"\n✅ BEST CONFIG: threshold={best_config[0]}, dilation_iter={best_config[1]}")
    print(f"   Dice = {best_dice:.4f}")

In [ ]:
# =============================================================================
# VISUALIZE BEST DILATION RESULT
# =============================================================================

if VOLUME_ID is not None and best_config is not None:
    print(f"Visualizing best config: threshold={best_config[0]}, dilation={best_config[1]}")
    
    # Apply best config
    best_thr, best_dil = best_config
    pred_best = (pred_prob > best_thr).astype(np.uint8)
    if best_dil > 0:
        pred_best = apply_dilation_2d_per_slice(pred_best, iterations=best_dil)
    
    # Also show original (threshold=0.4, no dilation) for comparison
    pred_original = (pred_prob > 0.4).astype(np.uint8)
    
    # Visualize multiple slices
    slice_indices = [ct_vol.shape[0] // 4, ct_vol.shape[0] // 2, 3 * ct_vol.shape[0] // 4]
    
    fig, axes = plt.subplots(len(slice_indices), 4, figsize=(20, 5 * len(slice_indices)))
    
    for i, slice_idx in enumerate(slice_indices):
        ct_slice = ct_vol[slice_idx]
        gt_slice = gt_vol[slice_idx]
        
        # GT
        axes[i, 0].imshow(ct_slice, cmap='gray')
        axes[i, 0].imshow(gt_slice, cmap='Greens', alpha=0.5 * (gt_slice > 0))
        axes[i, 0].set_title(f'Ground Truth (z={slice_idx})')
        axes[i, 0].axis('off')
        
        # Original prediction
        overlay_orig = create_overlay_image(ct_slice, gt_slice, pred_original[slice_idx])
        axes[i, 1].imshow(overlay_orig)
        axes[i, 1].set_title('Original (thr=0.4, no dilation)')
        axes[i, 1].axis('off')
        
        # Best prediction
        overlay_best = create_overlay_image(ct_slice, gt_slice, pred_best[slice_idx])
        axes[i, 2].imshow(overlay_best)
        axes[i, 2].set_title(f'Best (thr={best_thr}, dil={best_dil})')
        axes[i, 2].axis('off')
        
        # Side by side prediction comparison
        axes[i, 3].imshow(ct_slice, cmap='gray')
        axes[i, 3].imshow(pred_best[slice_idx], cmap='Oranges', alpha=0.5 * (pred_best[slice_idx] > 0))
        axes[i, 3].set_title('Best Prediction Mask')
        axes[i, 3].axis('off')
    
    plt.suptitle(f"Comparison: Original vs Best Config (thr={best_thr}, dilation={best_dil})", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Print final metrics comparison
    valid_mask = gt_vol != 2
    gt_fg = (gt_vol == 1) & valid_mask
    
    # Original metrics
    pred_orig_fg = (pred_original == 1) & valid_mask
    tp_orig = (gt_fg & pred_orig_fg).sum()
    fn_orig = (gt_fg & ~pred_orig_fg).sum()
    fp_orig = (~gt_fg & pred_orig_fg).sum()
    dice_orig = (2 * tp_orig) / (2 * tp_orig + fn_orig + fp_orig + 1e-8)
    
    # Best metrics
    pred_best_fg = (pred_best == 1) & valid_mask
    tp_best = (gt_fg & pred_best_fg).sum()
    fn_best = (gt_fg & ~pred_best_fg).sum()
    fp_best = (~gt_fg & pred_best_fg).sum()
    dice_best = (2 * tp_best) / (2 * tp_best + fn_best + fp_best + 1e-8)
    prec_best = tp_best / (tp_best + fp_best + 1e-8)
    rec_best = tp_best / (tp_best + fn_best + 1e-8)
    
    print("\n" + "="*60)
    print("IMPROVEMENT SUMMARY")
    print("="*60)
    print(f"Original (thr=0.4, no dilation): Dice = {dice_orig:.4f}")
    print(f"Best (thr={best_thr}, dil={best_dil}):     Dice = {dice_best:.4f}")
    print(f"")
    print(f"Improvement: +{(dice_best - dice_orig):.4f} ({100*(dice_best - dice_orig)/dice_orig:.1f}%)")
    print(f"")
    print(f"Best config metrics:")
    print(f"  Precision: {prec_best:.4f}")
    print(f"  Recall:    {rec_best:.4f}")
    print("="*60)